# TinyGPT v2 — Interactive Playground + Automated Benchmark

This notebook **loads existing checkpoints only**. It does **not** retrain models or export GGUF.

## What it does
- Keeps the interactive chat playground with Mix A / Mix B / optional Stage 4 Base, history toggle, and generation controls.
- Runs a reproducible, isolated benchmark over identity, conversation, empathy, factual knowledge, reasoning, coding, hallucination-resistance, instruction-following, and short-term recall prompts.
- Evaluates several decoding configurations with fresh model inputs for every prompt and saves TXT, JSON, and CSV reports.
- Records simple repetition / identical-answer warnings as diagnostics, **not** as a substitute for human correctness scoring.

## Kaggle setup
Run cells from the top in a **fresh Kaggle session**. The first code cell pins Transformers/tokenizers/huggingface-hub to versions compatible with the checkpoint-loading code and avoids the reported Transformers 5.x / PyTorch distributed-import crash. If the old failed import has already been attempted in this kernel, use **Restart Session** before running all cells.

Attach the saved output from your current `stage4/sft` run (Mix A and Mix B best checkpoints). Attach the `pretrain-stage-4` output if you want an optional base-model comparison. Enable GPU and Internet if the GPT-2 tokenizer isn't cached.

Outputs: `/kaggle/working/stage4_sft_interactive_benchmark_20261024/`

**No GGUF conversion is performed in this notebook.**

In [15]:
# CELL 1 — Repair a Transformers/PyTorch incompatibility BEFORE importing either library.
# The reported Kaggle traceback comes from Transformers 5.x importing newer distributed
# sharding code that conflicts with the installed PyTorch 2.11 runtime. This notebook uses
# the established Transformers 4.x GPT-2 implementation that matches our checkpoint format.
import importlib.metadata as _metadata
import subprocess as _subprocess
import sys as _sys

_REQUIRED_PACKAGES = {
    "transformers": "4.57.3",
    "huggingface-hub": "0.36.0",
    "tokenizers": "0.22.1",
}

def _installed_version(package_name):
    try:
        return _metadata.version(package_name)
    except _metadata.PackageNotFoundError:
        return None

_current_versions = {name: _installed_version(name) for name in _REQUIRED_PACKAGES}
print("Installed HF stack before check:", _current_versions)
_needs_repair = any(_current_versions[name] != version for name, version in _REQUIRED_PACKAGES.items())

if _needs_repair:
    # A prior failed Transformers import can leave partially imported modules and PyTorch
    # registrations in this kernel. Updating packages in that same live process is unsafe.
    _transformers_already_loaded = any(
        name == "transformers" or name.startswith("transformers.") for name in _sys.modules
    )
    if _transformers_already_loaded:
        raise RuntimeError(
            "This kernel already attempted to import an incompatible Transformers build. "
            "Use Kaggle's Restart Session action, then Run All from the first cell. "
            "The new clean session will install the pinned compatible versions before imports."
        )

    _pip_command = [
        _sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
        "transformers==4.57.3", "huggingface-hub==0.36.0", "tokenizers==0.22.1",
    ]
    print("Installing a compatible Transformers 4.x stack (PyTorch itself will not be replaced)...")
    _subprocess.check_call(_pip_command)

_final_versions = {name: _installed_version(name) for name in _REQUIRED_PACKAGES}
print("HF stack after check:", _final_versions)
if any(_final_versions[name] != version for name, version in _REQUIRED_PACKAGES.items()):
    raise RuntimeError(f"Package version verification failed: {_final_versions}")

# Smoke-test the exact imports that failed in the previous run, before checkpoint discovery.
# This should fail early with a targeted message if the Kaggle image still has a broken stack.
try:
    import torch as _torch_preflight
    from transformers import GPT2Config as _GPT2Config_preflight
    from transformers import GPT2LMHeadModel as _GPT2LMHeadModel_preflight
    from transformers import AutoTokenizer as _AutoTokenizer_preflight
except Exception as _import_error:
    raise RuntimeError(
        "The pinned Transformers/PyTorch import smoke test failed before loading any model weights. "
        "If this cell is being rerun after an earlier failed Transformers import, use Kaggle's "
        "Restart Session and then Run All from the top. Original error: "
        f"{type(_import_error).__name__}: {_import_error}"
    ) from _import_error

print("Import smoke test passed:", _torch_preflight.__version__, "| Transformers", _final_versions['transformers'])
print("Compatibility preflight passed. Continue to the next cell.")


Installed HF stack before check: {'transformers': '4.57.3', 'huggingface-hub': '0.36.0', 'tokenizers': '0.22.1'}
HF stack after check: {'transformers': '4.57.3', 'huggingface-hub': '0.36.0', 'tokenizers': '0.22.1'}
Import smoke test passed: 2.11.0+cu128 | Transformers 4.57.3
Compatibility preflight passed. Continue to the next cell.


In [16]:
# CELL 2 — Setup and checkpoint discovery (compatible HF stack preinstalled)
import os, json, time, random, re, csv, hashlib, traceback
from pathlib import Path
from datetime import datetime, timezone
import torch
from transformers import GPT2Config, GPT2LMHeadModel, AutoTokenizer

SEED = 20261024
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/stage4_sft_interactive_benchmark_20261024")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Preferred source locations. If a preferred path is absent, discovery searches INPUT_ROOT.
PREFERRED_BASE_CHECKPOINT = Path("/kaggle/input/notebooks/fakesiddharth/pretrain-stage-4/checkpoints/stage4_best.pt")
PREFERRED_SFT_ROOT = Path("/kaggle/input/notebooks/fakesiddharth/stage4-sft")
SFT_RUN_DIRNAME = "stage4_sft_run_20261023"

# Fill in an exact path here only if Kaggle has multiple copies and reports ambiguity.
EXPLICIT_CHECKPOINTS = {
    "BASE_STAGE4": None,
    "MIX_A": None,
    "MIX_B": None,
}
INCLUDE_BASE_IN_AUTOMATED_BENCHMARK = False  # set True to benchmark base too, if attached

print("PyTorch:", torch.__version__, "| Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), "| GPU count:", torch.cuda.device_count())

try:
    tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
except Exception as exc:
    raise RuntimeError(
        "Could not load the GPT-2 tokenizer. Enable Internet in Kaggle or attach/cache the tokenizer. "
        f"Original error: {type(exc).__name__}: {exc}"
    ) from exc
if len(tokenizer) != 50257 or tokenizer.eos_token_id != 50256:
    raise RuntimeError(f"Unexpected GPT-2 tokenizer: vocab={len(tokenizer)}, EOS={tokenizer.eos_token_id}")
EOS = tokenizer.eos_token_id
tokenizer.pad_token = tokenizer.eos_token

MODEL_CONFIG = GPT2Config(
    vocab_size=50257, n_positions=256, n_ctx=256, n_embd=512,
    n_layer=8, n_head=8, resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0,
    bos_token_id=EOS, eos_token_id=EOS, pad_token_id=EOS, use_cache=True,
)
EXPECTED_PARAMS = 51_082_752

def find_files(filename, root=INPUT_ROOT):
    root = Path(root)
    if not root.exists():
        return []
    return sorted({p.resolve() for p in root.rglob(filename) if p.is_file()})

def choose_unambiguous(paths, label, required=True):
    paths = sorted({Path(p).resolve() for p in paths})
    if len(paths) == 1:
        return paths[0]
    if len(paths) == 0 and not required:
        return None
    if not paths:
        raise FileNotFoundError(f"Could not find {label} under {INPUT_ROOT}. Attach the relevant Kaggle output.")
    listing = "\n".join(f"  - {p}" for p in paths[:30])
    raise RuntimeError(
        f"Found {len(paths)} candidate files for {label}; refusing to choose an arbitrary checkpoint:\n{listing}\n"
        "Set EXPLICIT_CHECKPOINTS in Cell 1 to the exact desired path, or detach old/duplicate outputs."
    )

def discover_checkpoint(label, filename, preferred_path=None, required=True):
    explicit = EXPLICIT_CHECKPOINTS.get(label)
    if explicit:
        p = Path(explicit)
        if not p.is_file():
            raise FileNotFoundError(f"Explicit checkpoint path for {label} does not exist: {p}")
        return p.resolve()
    if preferred_path is not None and Path(preferred_path).is_file():
        return Path(preferred_path).resolve()
    matches = find_files(filename)
    lower_label = label.lower()
    if label == "BASE_STAGE4":
        preferred = [p for p in matches if "pretrain-stage-4" in str(p).lower() and "checkpoints" in str(p).lower()]
        if preferred:
            matches = preferred
        else:
            matches = [p for p in matches if "stage4" in str(p).lower() or "pretrain" in str(p).lower()]
    else:
        # Prefer artifacts from the named run if present; otherwise constrain to the SFT output root.
        current_run = [p for p in matches if SFT_RUN_DIRNAME.lower() in str(p).lower()]
        if current_run:
            matches = current_run
        else:
            sft_root = str(PREFERRED_SFT_ROOT).lower()
            rooted = [p for p in matches if sft_root in str(p).lower() or "stage4_sft_run" in str(p).lower()]
            if rooted:
                matches = rooted
    return choose_unambiguous(matches, label, required=required)

BASE_PATH = discover_checkpoint("BASE_STAGE4", "stage4_best.pt", PREFERRED_BASE_CHECKPOINT, required=False)
MIXA_PATH = discover_checkpoint("MIX_A", "sft_mixA_best.pt", required=True)
MIXB_PATH = discover_checkpoint("MIX_B", "sft_mixB_best.pt", required=True)
CHECKPOINTS = {"Mix A": MIXA_PATH, "Mix B": MIXB_PATH}
if BASE_PATH is not None:
    CHECKPOINTS["Stage 4 Base"] = BASE_PATH

print("\nSelected checkpoint files:")
for label, path in CHECKPOINTS.items():
    print(f"{label}: {path}")
print("Stage 4 Base:", BASE_PATH if BASE_PATH is not None else "not attached (optional)")
print("Output directory:", OUTPUT_DIR.resolve())

PyTorch: 2.11.0+cu128 | Device: cuda
GPU: Tesla T4 | GPU count: 2

Selected checkpoint files:
Mix A: /kaggle/input/notebooks/fakesiddharth/stage4-sft/stage4_sft_run_20261023/mixA/sft_mixA_best.pt
Mix B: /kaggle/input/notebooks/fakesiddharth/stage4-sft/stage4_sft_run_20261023/mixB/sft_mixB_best.pt
Stage 4 Base: not attached (optional)
Output directory: /kaggle/working/stage4_sft_interactive_benchmark_20261024


In [17]:
# CELL 3 — Load and strictly verify models

def unwrap_state(payload, path):
    if not isinstance(payload, dict):
        raise TypeError(f"Unexpected checkpoint object in {path}: {type(payload).__name__}")
    state = payload
    for key in ("model_state_dict", "state_dict", "model", "model_weights", "weights"):
        value = payload.get(key)
        if isinstance(value, dict) and value and all(torch.is_tensor(v) for v in value.values()):
            state = value
            break
    if not state or not all(torch.is_tensor(v) for v in state.values()):
        raise RuntimeError(f"No raw tensor state_dict found in {path}")
    if all(k.startswith("module.") for k in state):
        state = {k[len("module."):]: v for k, v in state.items()}
    return state


def load_model(path, label):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"{label} checkpoint missing: {path}")
    try:
        payload = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    state = unwrap_state(payload, path)
    model = GPT2LMHeadModel(MODEL_CONFIG)
    incompatible = model.load_state_dict(state, strict=False)
    if incompatible.missing_keys or incompatible.unexpected_keys:
        raise RuntimeError(
            f"{label} architecture mismatch. Missing={incompatible.missing_keys[:8]}, "
            f"unexpected={incompatible.unexpected_keys[:8]}"
        )
    n_params = sum(p.numel() for p in model.parameters())
    if n_params != EXPECTED_PARAMS:
        raise RuntimeError(f"{label} has {n_params:,} parameters; expected {EXPECTED_PARAMS:,}")
    model.to(DEVICE).eval()
    print(f"Loaded {label}: {n_params:,} parameters | {path.stat().st_size / (1024**2):.1f} MiB")
    del payload, state
    return model

models = {}
for label, path in CHECKPOINTS.items():
    if path is not None:
        models[label] = load_model(path, label)
if not ("Mix A" in models and "Mix B" in models):
    raise RuntimeError("Both Mix A and Mix B are required for testing.")
print("\nReady models:", ", ".join(models.keys()))

Loaded Mix A: 51,082,752 parameters | 194.9 MiB
Loaded Mix B: 51,082,752 parameters | 194.9 MiB

Ready models: Mix A, Mix B


In [18]:
# CELL 4 — Shared generation utilities (fresh inputs; no cross-request KV cache)
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

MAX_CONTEXT = MODEL_CONFIG.n_positions

def clean_user_text(text):
    text = str(text).strip()
    text = re.sub(r"^(?:User|Assistant):\s*", "", text, flags=re.I).strip()
    return text

def seed_everything(seed):
    seed = int(seed) % (2**31 - 1)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)

def build_prompt(user_text, history=None, keep_history=False, max_prompt_tokens=200):
    """Build the SFT role format and drop only whole older turns when history is enabled."""
    current_text = clean_user_text(user_text)
    current = f"User: {current_text}\nAssistant:"
    if not keep_history or not history:
        candidate = current
    else:
        usable = list(history)
        candidate = current
        while usable:
            chunks = [f"User: {u}\nAssistant: {a}" for u, a in usable] + [current]
            candidate = "\n".join(chunks)
            if len(tokenizer.encode(candidate, add_special_tokens=False)) <= max_prompt_tokens:
                break
            usable.pop(0)
        else:
            candidate = current
    ids = tokenizer.encode(candidate, add_special_tokens=False)
    if len(ids) > max_prompt_tokens:
        current_ids = tokenizer.encode(current_text, add_special_tokens=False)
        safe_text = tokenizer.decode(current_ids[:max(1, max_prompt_tokens - 8)], skip_special_tokens=True)
        candidate = f"User: {safe_text}\nAssistant:"
    return candidate

def cut_at_stop_markers(text):
    # Stop if the model starts a new role/turn instead of continuing the answer.
    stops = ["\nUser:", "\nAssistant:", "<|endoftext|>"]
    cut = len(text)
    for marker in stops:
        at = text.find(marker)
        if at >= 0:
            cut = min(cut, at)
    return text[:cut].strip()

@torch.inference_mode()
def generate_from_prompt(model, prompt, *, do_sample=False, temperature=0.7, top_p=0.9,
                         top_k=50, repetition_penalty=1.0, max_new_tokens=96, seed=SEED):
    """Every invocation tokenizes a fresh prompt and calls generate without a reused cache."""
    prompt = str(prompt).strip()
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    ids = enc["input_ids"]
    # Keep context below 256 tokens while preserving room for the requested answer.
    max_prompt = max(1, MAX_CONTEXT - max(1, int(max_new_tokens)))
    if ids.shape[1] > max_prompt:
        ids = ids[:, -max_prompt:]
    prompt_len = int(ids.shape[1])
    input_ids = ids.to(DEVICE)
    attention_mask = torch.ones_like(input_ids, device=DEVICE)
    new_tokens = min(max(1, int(max_new_tokens)), MAX_CONTEXT - prompt_len)
    kwargs = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "max_new_tokens": new_tokens,
        "do_sample": bool(do_sample),
        "eos_token_id": EOS,
        "pad_token_id": EOS,
        "repetition_penalty": float(repetition_penalty),
        "use_cache": True,
    }
    if do_sample:
        if temperature <= 0:
            raise ValueError("Temperature must be > 0 when sampling is enabled.")
        kwargs.update({
            "temperature": float(temperature),
            "top_p": float(top_p),
            "top_k": int(top_k) if int(top_k) > 0 else 0,
        })
    seed_everything(seed)
    model.eval()
    start = time.perf_counter()
    output_ids = model.generate(**kwargs)
    elapsed = time.perf_counter() - start
    generated_ids = output_ids[0, prompt_len:].detach().cpu()
    raw = tokenizer.decode(generated_ids, skip_special_tokens=False)
    response = cut_at_stop_markers(raw)
    del output_ids, generated_ids
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return (response if response else "<empty response>"), elapsed

def make_user_prompt(text, history=None, keep_history=False):
    # Automated cases may be supplied as already-formatted multi-turn context.
    if str(text).lstrip().startswith("User:") and "\nAssistant:" in str(text):
        return str(text).strip()
    return build_prompt(text, history=history, keep_history=keep_history)

def response_diagnostics(text):
    """Small heuristic flags only; they do not determine factual correctness."""
    s = re.sub(r"\s+", " ", str(text)).strip().lower()
    flags = []
    if not s or s == "<empty response>":
        flags.append("empty")
    words = re.findall(r"\b[\w'-]+\b", s)
    if len(words) >= 12:
        for n in (3, 4, 5):
            grams = [tuple(words[i:i+n]) for i in range(len(words)-n+1)]
            if len(grams) >= n and len(set(grams)) <= max(1, len(grams) * 0.6):
                flags.append("repeated_ngrams")
                break
    return sorted(set(flags))

In [19]:
# CELL 5 — Automated benchmark across prompts and decoding configurations
# This cell RUNS the benchmark and writes TXT, JSON, and CSV reports.
BENCHMARK_PROMPTS = [
    ("identity", "Hi! How are you today?"),
    ("identity", "Who are you, and what is your purpose? If you don't know who created you, don't invent an answer."),
    ("identity", "Are you a human or an AI language model? Explain briefly."),
    ("empathy", "I've been studying for my exams all day, and I'm feeling overwhelmed. Can you help me feel a little better?"),
    ("empathy", "I'm excited because I finally understood a difficult programming concept!"),
    ("empathy", "I had a really difficult day today."),
    ("conversation", "Can you help me think through a problem? I'm not sure where to start."),
    ("science", "What is the capital of France? Answer in one sentence."),
    ("science", "What is photosynthesis? Explain it in simple terms in no more than three sentences."),
    ("science", "Explain what a GPU does to someone who has never studied computers."),
    ("everyday_knowledge", "What are three benefits of regular exercise? Give exactly three concise points."),
    ("everyday_knowledge", "What are two differences between an HDD and an SSD? Give exactly two numbered points."),
    ("math", "A person has 12 apples, buys 8 more, and gives 5 away. How many remain? Show the calculation."),
    ("logic", "All roses are flowers. Some flowers fade quickly. Does this prove that some roses fade quickly? Explain your reasoning."),
    ("math", "A notebook costs ₹40. How much do 3 notebooks cost? Explain in one sentence."),
    ("coding", "Write a Python function named square that takes a number and returns its square. Include one example call."),
    ("coding", "What is the difference between a Python list and a Python tuple? Explain in two bullet points."),
    ("coding", "Fix the bug: def add(a, b): return a - b. The function should add two numbers."),
    ("coding", "Write a tiny Python program that prints a greeting."),
    ("hallucination_resistance", "What is the exact temperature inside my room right now? Give the temperature in Celsius."),
    ("hallucination_resistance", "Explain why the Sun revolves around the Earth once every 24 hours."),
    ("honesty", "Can you guarantee that every answer you give me is factually correct? Explain briefly."),
    ("instruction_following", "Name exactly three fruits. Separate them with commas. Do not write anything else."),
    ("instruction_following", "Explain what a CPU is using exactly 15 words."),
    ("summarization", "Summarize in one sentence: A computer stores and processes information. The CPU executes instructions, RAM temporarily holds data used by active programs, and storage preserves files when power is off."),
    ("context_recall", "User: My favourite programming language is Python.\nAssistant: Nice!\nUser: Which programming language did I say was my favourite?\nAssistant:"),
    ("context_recall", "User: My name is Alex and I live in Oslo.\nAssistant: Nice to meet you, Alex.\nUser: Which city did I say I live in?\nAssistant:"),
]

# Fixed, documented configurations. All prompt/model combinations are run from fresh inputs.
DECODE_CONFIGS = [
    {"name":"greedy_neutral", "do_sample":False, "temperature":None, "top_p":1.0, "top_k":0, "repetition_penalty":1.0, "max_new_tokens":96},
    {"name":"greedy_repeat_penalty", "do_sample":False, "temperature":None, "top_p":1.0, "top_k":0, "repetition_penalty":1.15, "max_new_tokens":96},
    {"name":"sample_balanced", "do_sample":True, "temperature":0.7, "top_p":0.9, "top_k":50, "repetition_penalty":1.10, "max_new_tokens":96},
    {"name":"sample_conservative", "do_sample":True, "temperature":0.5, "top_p":0.85, "top_k":40, "repetition_penalty":1.15, "max_new_tokens":96},
]

# Primary comparison uses the two SFT models. Set INCLUDE_BASE_IN_AUTOMATED_BENCHMARK=True in Cell 1 to add base.
BENCHMARK_MODEL_NAMES = ["Mix A", "Mix B"]
if INCLUDE_BASE_IN_AUTOMATED_BENCHMARK and "Stage 4 Base" in models:
    BENCHMARK_MODEL_NAMES.append("Stage 4 Base")

def stable_test_seed(prompt_idx, config_name):
    raw = f"{SEED}|{prompt_idx}|{config_name}".encode("utf-8")
    return int(hashlib.sha256(raw).hexdigest()[:8], 16) % (2**31 - 1)

def run_automated_benchmark():
    started = datetime.now(timezone.utc)
    results = []
    total = len(BENCHMARK_PROMPTS) * len(DECODE_CONFIGS) * len(BENCHMARK_MODEL_NAMES)
    print(f"Starting isolated benchmark: {len(BENCHMARK_PROMPTS)} prompts × {len(DECODE_CONFIGS)} configs × {len(BENCHMARK_MODEL_NAMES)} models = {total} generations")
    print("Each call builds fresh input_ids and does not pass conversation history or a prior KV cache.")
    done = 0
    for cfg in DECODE_CONFIGS:
        print(f"\n=== Configuration: {cfg['name']} ===")
        for p_idx, (category, prompt_text) in enumerate(BENCHMARK_PROMPTS, start=1):
            # The two context-recall cases intentionally provide their own self-contained context.
            formatted = make_user_prompt(prompt_text, keep_history=False)
            record = {
                "prompt_id": p_idx, "category": category, "prompt": prompt_text,
                "formatted_prompt": formatted, "config": dict(cfg), "responses": {}
            }
            for model_name in BENCHMARK_MODEL_NAMES:
                if model_name not in models:
                    continue
                try:
                    seed = stable_test_seed(p_idx, cfg["name"])
                    response, seconds = generate_from_prompt(
                        models[model_name], formatted,
                        do_sample=cfg["do_sample"],
                        temperature=cfg["temperature"] if cfg["temperature"] is not None else 0.7,
                        top_p=cfg["top_p"], top_k=cfg["top_k"],
                        repetition_penalty=cfg["repetition_penalty"],
                        max_new_tokens=cfg["max_new_tokens"], seed=seed,
                    )
                    record["responses"][model_name] = {
                        "text": response, "seconds": round(seconds, 4),
                        "diagnostic_flags": response_diagnostics(response), "seed": seed,
                    }
                except Exception as exc:
                    record["responses"][model_name] = {
                        "text": f"<ERROR {type(exc).__name__}: {exc}>",
                        "seconds": None, "diagnostic_flags":["generation_error"],
                    }
                done += 1
                if done % 20 == 0 or done == total:
                    print(f"Progress: {done}/{total} generations")
            results.append(record)
            # Save a lightweight progress checkpoint periodically so partial results survive a runtime interruption.
            if p_idx % 3 == 0 or p_idx == len(BENCHMARK_PROMPTS):
                partial_path = OUTPUT_DIR / "benchmark_results_partial.json"
                partial_path.write_text(json.dumps({
                    "status": "running",
                    "updated_utc": datetime.now(timezone.utc).isoformat(),
                    "completed_generation_count": done,
                    "expected_generation_count": total,
                    "results": results,
                }, ensure_ascii=False, indent=2), encoding="utf-8")
    completed = datetime.now(timezone.utc)

    # Add within-model/config repeated-answer diagnostics. This is only a heuristic.
    duplicate_counts = {}
    for cfg in DECODE_CONFIGS:
        for model_name in BENCHMARK_MODEL_NAMES:
            keys = []
            for rec in results:
                if rec["config"]["name"] != cfg["name"]:
                    continue
                item = rec["responses"].get(model_name, {})
                text = re.sub(r"\s+", " ", item.get("text", "")).strip().lower()
                if text and not text.startswith("<error"):
                    keys.append(text)
            duplicate_counts[f"{model_name}|{cfg['name']}"] = len(keys) - len(set(keys))

    json_path = OUTPUT_DIR / "benchmark_results.json"
    txt_path = OUTPUT_DIR / "benchmark_report.txt"
    csv_path = OUTPUT_DIR / "benchmark_results.csv"
    summary_path = OUTPUT_DIR / "benchmark_summary.json"
    json_path.write_text(json.dumps({
        "started_utc": started.isoformat(), "completed_utc": completed.isoformat(),
        "models": BENCHMARK_MODEL_NAMES, "decode_configs": DECODE_CONFIGS,
        "prompt_count": len(BENCHMARK_PROMPTS), "result_count": len(results),
        "duplicate_response_counts_by_model_config": duplicate_counts,
        "results": results,
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    with txt_path.open("w", encoding="utf-8") as f:
        f.write("TINYGPT v2 — ISOLATED MULTI-CONFIG BENCHMARK REPORT\n")
        f.write("=" * 88 + "\n")
        f.write(f"Started UTC: {started.isoformat()}\nCompleted UTC: {completed.isoformat()}\n")
        f.write(f"Models: {', '.join(BENCHMARK_MODEL_NAMES)}\n")
        f.write(f"Prompts: {len(BENCHMARK_PROMPTS)} | Configurations: {len(DECODE_CONFIGS)}\n")
        f.write("Isolation: each response receives freshly tokenized input; no playground history or previous KV cache is passed.\n")
        f.write("Heuristic repetition flags are diagnostics only, not correctness scores.\n\n")
        for cfg in DECODE_CONFIGS:
            f.write("\n" + "#" * 88 + f"\nCONFIG: {cfg['name']} | {cfg}\n" + "#" * 88 + "\n")
            for rec in results:
                if rec["config"]["name"] != cfg["name"]:
                    continue
                f.write(f"\nPROMPT {rec['prompt_id']:02d} [{rec['category']}]: {rec['prompt']}\n")
                for model_name, response in rec["responses"].items():
                    f.write(f"\n--- {model_name} | {response.get('seconds')}s | flags={response.get('diagnostic_flags', [])} ---\n")
                    f.write(response.get("text", "") + "\n")
                f.write("-" * 88 + "\n")
        f.write("\nDUPLICATE-RESPONSE HEURISTICS (same normalized text reused for different prompts)\n")
        for key, count in duplicate_counts.items():
            f.write(f"{key}: {count}\n")

    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["prompt_id", "category", "prompt", "config", "model", "response", "seconds", "diagnostic_flags", "seed"])
        writer.writeheader()
        for rec in results:
            for model_name, response in rec["responses"].items():
                writer.writerow({
                    "prompt_id": rec["prompt_id"], "category": rec["category"], "prompt": rec["prompt"],
                    "config": rec["config"]["name"], "model": model_name,
                    "response": response.get("text", ""), "seconds": response.get("seconds"),
                    "diagnostic_flags": ",".join(response.get("diagnostic_flags", [])), "seed": response.get("seed"),
                })

    summary = {
        "started_utc": started.isoformat(), "completed_utc": completed.isoformat(),
        "models": BENCHMARK_MODEL_NAMES, "prompt_count": len(BENCHMARK_PROMPTS),
        "configurations": [x["name"] for x in DECODE_CONFIGS],
        "total_generations": total, "duplicate_response_counts_by_model_config": duplicate_counts,
        "report_txt": str(txt_path), "report_json": str(json_path), "report_csv": str(csv_path),
    }
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\nBENCHMARK COMPLETE")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return results

benchmark_results = run_automated_benchmark()

Starting isolated benchmark: 27 prompts × 4 configs × 2 models = 216 generations
Each call builds fresh input_ids and does not pass conversation history or a prior KV cache.

=== Configuration: greedy_neutral ===
Progress: 20/216 generations
Progress: 40/216 generations

=== Configuration: greedy_repeat_penalty ===
Progress: 60/216 generations
Progress: 80/216 generations
Progress: 100/216 generations

=== Configuration: sample_balanced ===
Progress: 120/216 generations
Progress: 140/216 generations
Progress: 160/216 generations

=== Configuration: sample_conservative ===
Progress: 180/216 generations
Progress: 200/216 generations
Progress: 216/216 generations

BENCHMARK COMPLETE
{
  "started_utc": "2026-10-24T14:16:10.396992+00:00",
  "completed_utc": "2026-10-24T14:16:43.624806+00:00",
  "models": [
    "Mix A",
    "Mix B"
  ],
  "prompt_count": 27,
  "configurations": [
    "greedy_neutral",
    "greedy_repeat_penalty",
    "sample_balanced",
    "sample_conservative"
  ],
  "total

In [20]:
# CELL 6 — Interactive Playground (manual prompts, model comparison, optional chat history)
try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown, clear_output
except Exception as exc:
    raise RuntimeError(f"ipywidgets is unavailable. Enable notebook widgets in Kaggle. Original error: {exc}") from exc

model_options = list(models.keys())
if "Mix A" in models and "Mix B" in models:
    model_options.append("Compare Mix A vs Mix B")
if len(models) > 2:
    model_options.append("Compare all loaded models")

model_picker = widgets.Dropdown(options=model_options, value="Compare Mix A vs Mix B" if "Compare Mix A vs Mix B" in model_options else model_options[0], description="Model:", layout=widgets.Layout(width="400px"))
mode_picker = widgets.ToggleButtons(options=[("Greedy", "greedy"), ("Sampled", "sampled")], value="sampled", description="Decode:")
max_new_tokens_widget = widgets.IntSlider(value=96, min=8, max=160, step=8, description="Max new tokens:", continuous_update=False, layout=widgets.Layout(width="500px"))
temperature_widget = widgets.FloatSlider(value=0.7, min=0.1, max=1.5, step=0.05, description="Temperature:", readout_format=".2f", continuous_update=False, layout=widgets.Layout(width="500px"))
top_p_widget = widgets.FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description="Top-p:", readout_format=".2f", continuous_update=False, layout=widgets.Layout(width="500px"))
top_k_widget = widgets.IntSlider(value=50, min=0, max=100, step=5, description="Top-k (0=off):", continuous_update=False, layout=widgets.Layout(width="500px"))
rep_penalty_widget = widgets.FloatSlider(value=1.10, min=1.0, max=1.6, step=0.02, description="Repetition penalty:", readout_format=".2f", continuous_update=False, layout=widgets.Layout(width="500px"))
seed_widget = widgets.IntText(value=SEED, description="Seed:", layout=widgets.Layout(width="240px"))
prompt_box = widgets.Textarea(value="Hi! How are you today?", description="Your prompt:", placeholder="Ask TinyGPT anything…", layout=widgets.Layout(width="100%", height="100px"))
keep_history_widget = widgets.Checkbox(value=False, description="Keep multi-turn history (independent prompts by default)", indent=False)
generate_button = widgets.Button(description="Generate response", button_style="primary", icon="paper-plane")
clear_button = widgets.Button(description="Clear chat", icon="trash")
save_button = widgets.Button(description="Save transcript", icon="save")
status = widgets.HTML(value="<span style='color:#666'>Ready.</span>")
chat_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="10px", width="100%"))
histories = {name: [] for name in models}
transcript = []

def selected_models_for_playground():
    choice = model_picker.value
    if choice == "Compare Mix A vs Mix B":
        return [name for name in ("Mix A", "Mix B") if name in models]
    if choice == "Compare all loaded models":
        return list(models.keys())
    return [choice]

def on_generate(_):
    user_text = prompt_box.value.strip()
    if not user_text:
        status.value = "<span style='color:#b45309'>Type a prompt first.</span>"
        return
    generate_button.disabled = True
    status.value = "<span style='color:#666'>Generating…</span>"
    current_rows = []
    try:
        for model_name in selected_models_for_playground():
            prompt = build_prompt(user_text, history=histories.get(model_name, []), keep_history=keep_history_widget.value)
            response, elapsed = generate_from_prompt(
                models[model_name], prompt,
                do_sample=(mode_picker.value == "sampled"),
                temperature=temperature_widget.value, top_p=top_p_widget.value,
                top_k=top_k_widget.value, repetition_penalty=rep_penalty_widget.value,
                max_new_tokens=max_new_tokens_widget.value, seed=seed_widget.value,
            )
            current_rows.append({"model":model_name, "response":response, "seconds":round(elapsed, 4)})
            if keep_history_widget.value:
                histories[model_name].append((clean_user_text(user_text), response))
            transcript.append({
                "timestamp_utc":datetime.now(timezone.utc).isoformat(), "model":model_name,
                "prompt":user_text, "response":response, "mode":mode_picker.value,
                "max_new_tokens":max_new_tokens_widget.value, "temperature":temperature_widget.value,
                "top_p":top_p_widget.value, "top_k":top_k_widget.value,
                "repetition_penalty":rep_penalty_widget.value, "seed":seed_widget.value,
                "history_enabled":keep_history_widget.value, "seconds":round(elapsed, 4),
            })
        with chat_output:
            display(Markdown(f"### You\n{user_text}"))
            for row in current_rows:
                display(Markdown(f"**{row['model']}** · {row['seconds']} s\n\n{row['response']}"))
            display(Markdown("---"))
        status.value = f"<span style='color:green'>Completed {len(current_rows)} response(s).</span>"
    except Exception as exc:
        status.value = f"<span style='color:#b91c1c'>Error: {type(exc).__name__}: {exc}</span>"
        with chat_output:
            display(Markdown(f"**Generation error:** `{type(exc).__name__}: {exc}`"))
    finally:
        generate_button.disabled = False

def on_clear(_):
    for hist in histories.values():
        hist.clear()
    with chat_output:
        clear_output(wait=False)
    status.value = "<span style='color:green'>Chat cleared; histories reset.</span>"

def on_save(_):
    json_path = OUTPUT_DIR / "interactive_transcript.json"
    txt_path = OUTPUT_DIR / "interactive_transcript.txt"
    json_path.write_text(json.dumps(transcript, ensure_ascii=False, indent=2), encoding="utf-8")
    with txt_path.open("w", encoding="utf-8") as f:
        f.write("TinyGPT v2 — Interactive Playground Transcript\n")
        f.write(f"Saved UTC: {datetime.now(timezone.utc).isoformat()}\n\n")
        for i, row in enumerate(transcript, 1):
            f.write(f"[{i}] {row['model']} | mode={row['mode']} | history={row['history_enabled']} | {row['seconds']} s\n")
            f.write(f"Prompt: {row['prompt']}\nResponse: {row['response']}\n\n")
    status.value = f"<span style='color:green'>Saved transcript: {json_path} and {txt_path}</span>"

generate_button.on_click(on_generate)
clear_button.on_click(on_clear)
save_button.on_click(on_save)

controls_1 = widgets.HBox([model_picker, mode_picker])
controls_2 = widgets.VBox([max_new_tokens_widget, temperature_widget, top_p_widget, top_k_widget, rep_penalty_widget, seed_widget, keep_history_widget])
buttons = widgets.HBox([generate_button, clear_button, save_button])
display(Markdown("## TinyGPT Interactive Playground\nTry custom prompts, switch models, adjust decoding, or enable chat history. The automated benchmark above always runs with isolated prompts."))
display(controls_1, controls_2, prompt_box, buttons, status, chat_output)

## TinyGPT Interactive Playground
Try custom prompts, switch models, adjust decoding, or enable chat history. The automated benchmark above always runs with isolated prompts.

Textarea(value='Hi! How are you today?', description='Your prompt:', layout=Layout(height='100px', width='100%…

HTML(value="<span style='color:#666'>Ready.</span>")

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

## Output files

The automated benchmark runs automatically in Cell 5 and saves:
- `benchmark_report.txt` — readable report for every prompt/configuration/model
- `benchmark_results.json` — complete structured outputs and settings
- `benchmark_results.csv` — one row per generated response
- `benchmark_summary.json` — report paths and duplicate-response diagnostic counts

The playground's **Save transcript** button writes `interactive_transcript.txt` and `interactive_transcript.json`.

The benchmark's duplicate/repetition flags are heuristics, not accuracy scores. Review answers manually for correctness and relevance. This notebook deliberately does not export to Hugging Face or GGUF; conversion can be handled separately after selecting the final checkpoint.